# G07 · A2 Task 1 — Held-out OCR ground truth & evaluation

**Owner:** K. M. Mehemud Azad (2105014) · branch `a2/heldout-ocr` · base commit `45c3fc3`

**What this notebook does:** selects a small, representative set of real Pierce (1890) pages,
drafts their transcription from the team's Document AI output, lets YOU hand-verify every page
against the scan (Section 7 — the actual ground-truth work), then scores the repo's default
Tesseract pipeline against your verified labels (CER / WER / word-F1).

## Kaggle setup (do this before running)
1. **Add data** → search `cruelangelssprint/pierce-1890-figure-and-ocr-outputs` → attach.
2. Notebook settings → **Internet ON**.
3. Accelerator: **None** (CPU is enough — don't burn GPU quota).

## Rules baked in (Mahdi's brief + faculty)
- Document AI text is a **draft to correct**, never ground truth; its confidence is never a label.
- Real Pierce pages only; nothing synthetic.
- Once selected + labelled, these pages are **held-out forever** — nobody tunes on them.

## Transcription conventions (v1 — cite these in the report)
- Transcribe **exactly what is printed**: keep archaic spellings, `œ/æ` ligatures, punctuation.
- Long-s (ſ) → type `s`. End-of-line hyphens: keep as printed. Line breaks → new lines
  (scoring collapses whitespace, so line structure is for readability only).
- Include page headers/footers/page-numbers as printed.
- Truly unreadable span → `⍰` (one per unreadable word).


In [ ]:
# ── 1. Environment ────────────────────────────────────────────────────────────
import os, sys, subprocess

%pip install -q "pymupdf>=1.25.5,<1.26" "opencv-python-headless>=4.10,<5.0" "pytesseract>=0.3.13,<0.4" "pydantic>=2.7,<3.0" "pydantic-settings>=2.2,<3.0" "pyyaml>=6.0,<7.0"

def sh(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

if subprocess.run(["which", "tesseract"], capture_output=True).returncode != 0:
    sh("apt-get -qq update > /dev/null 2>&1 || true")
    sh("apt-get -qq install -y tesseract-ocr > /dev/null 2>&1")

REPO_URL = "https://github.com/smammahdi/doc-agent-G07.git"
PIN = "45c3fc3"  # tip of a2/pierce-kb-foundation — the base Mahdi's brief names
if not os.path.exists("/kaggle/working/repo"):
    sh(f"git clone -q {REPO_URL} /kaggle/working/repo")
sh(f"cd /kaggle/working/repo && git checkout -q {PIN} && git log --oneline -1")
sys.path.insert(0, "/kaggle/working/repo/src")

import fitz, cv2, pytesseract, pydantic  # noqa: E402
TESSERACT_VERSION = subprocess.run(["tesseract", "--version"], capture_output=True, text=True).stdout.splitlines()[0]
print("python     :", sys.version.split()[0])
print("tesseract  :", TESSERACT_VERSION)
print("pymupdf    :", getattr(fitz, "__version__", None) or fitz.version[0])
print("opencv     :", cv2.__version__)


In [ ]:
# ── 2. Corpus: download + byte-for-byte verification (mirrors scripts/get_data.sh) ──
import hashlib, urllib.request
from pathlib import Path

RAW = Path("/kaggle/working/data/raw"); RAW.mkdir(parents=True, exist_ok=True)
PDF = RAW / "pierce-peoples-common-sense-medical-adviser-1890.pdf"
URL = "https://archive.org/download/peoplescommonsen00pier/peoplescommonsen00pier.pdf"
EXPECTED_BYTES = 65311598
EXPECTED_SHA = "841b1feb55ff0aff5735c3aeb308eb52e217f91ae55c5d34e21feb6a640c8896"

def sha256(p, chunk=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

if not (PDF.exists() and PDF.stat().st_size == EXPECTED_BYTES and sha256(PDF) == EXPECTED_SHA):
    print("downloading ~65 MB from Internet Archive (1890 edition, item peoplescommonsen00pier) ...")
    urllib.request.urlretrieve(URL, PDF)
assert PDF.stat().st_size == EXPECTED_BYTES, f"size mismatch: {PDF.stat().st_size} != {EXPECTED_BYTES}"
assert sha256(PDF) == EXPECTED_SHA, "SHA-256 mismatch — wrong file/edition, stop"
print("corpus verified:", PDF)


In [ ]:
# ── 3. Team sidecars: auto-discover DocAI words + Chandra layout in the attached dataset ──
import json

def _sniff_keys(path, limit=5):
    keys = set()
    try:
        with open(path, encoding="utf-8") as f:
            for i, line in enumerate(f):
                line = line.strip()
                if not line:
                    continue
                row = json.loads(line)
                if isinstance(row, dict):
                    keys |= set(row)
                if i >= limit:
                    break
    except Exception:
        return set()
    return keys

docai_path = chandra_path = None
docai_size = chandra_size = 0
for root, _, files in os.walk("/kaggle/input"):
    for name in files:
        if not name.lower().endswith((".jsonl", ".json")):
            continue
        p = os.path.join(root, name)
        k = _sniff_keys(p); s = os.path.getsize(p)
        if {"page_id", "text", "bbox_norm"} <= k and s > docai_size:
            docai_path, docai_size = p, s
        elif {"bbox", "page_box"} <= k and s > chandra_size:
            chandra_path, chandra_size = p, s

print("DocAI words JSONL :", docai_path)
print("Chandra layout    :", chandra_path)
assert docai_path, "Document AI words file not found — is the Kaggle dataset attached?"

# Parse with the REPO'S OWN loaders (same validation as the pipeline)
from doc_agent.vision import ocr as ocr_mod        # noqa: E402
from doc_agent.vision import layout as layout_mod  # noqa: E402
ref_words = ocr_mod._load_reference_words(Path(docai_path))
chandra = layout_mod._load_chandra(Path(chandra_path)) if chandra_path else {}
print(f"DocAI  : {sum(map(len, ref_words.values())):,} words on {len(ref_words):,} pages")
print(f"Chandra: {sum(map(len, chandra.values())):,} blocks on {len(chandra):,} pages")


In [ ]:
# ── 4. Page census (no rendering yet): sizes, rotation, word counts, figure counts ──
doc = fitz.open(str(PDF))
N = len(doc)
census = []
for i in range(N):
    pg = doc[i]; r = pg.rect
    pid = f"p{i + 1:04d}"
    census.append(dict(
        pid=pid, idx=i, w=round(r.width), h=round(r.height),
        landscape=(r.width > r.height) or pg.rotation in (90, 270),
        words=len(ref_words.get(pid, [])),
        figures=sum(1 for b in chandra.get(pid, []) if b["kind"] == "figure"),
    ))
ws = [c["words"] for c in census]
print(f"pages={N}  zero-word={sum(w == 0 for w in ws)}  under-20-words={sum(w < 20 for w in ws)}  "
      f"landscape={sum(c['landscape'] for c in census)}  figure-pages={sum(c['figures'] > 0 for c in census)}")


In [ ]:
# ── 5. Deterministic held-out selection (target 12 pages) ─────────────────────
SELECTION_RULE = """Held-out selection rule (deterministic, from sidecar census only —
no OCR quality signal is consulted, so selection cannot favour easy pages):
  1. figure pages ×2: most Chandra figure blocks (ties → lower page id), word-bearing;
  2. low/zero-word pages ×2: the two lowest page ids with <20 DocAI words (front-matter band);
  3. landscape ×1: lowest-id landscape/rotated page, if the book has one;
  4. body text ×remaining (to 12): pages with ≥150 words, no figures, portrait —
     picked at evenly spaced indices across that ordered list (stride sampling),
     covering the book start-to-end."""

def _spread(lst, k):
    if k <= 0 or len(lst) <= k:
        return lst[:max(k, 0)]
    return [lst[round(j * (len(lst) - 1) / (k - 1))] for j in range(k)]

figs = sorted((c for c in census if c["figures"] > 0 and c["words"] > 0),
              key=lambda c: (-c["figures"], c["pid"]))
low = sorted((c for c in census if c["words"] < 20), key=lambda c: c["pid"])
land = sorted((c for c in census if c["landscape"]), key=lambda c: c["pid"])
body = [c for c in census if c["words"] >= 150 and c["figures"] == 0 and not c["landscape"]]

proposal = []
def _add(cands, cat):
    for c in cands:
        if all(p["pid"] != c["pid"] for p in proposal):
            proposal.append({**c, "cat": cat})

_add(figs[:2], "figure"); _add(low[:2], "low/zero-word"); _add(land[:1], "landscape")
_add(_spread(body, 12 - len(proposal)), "body")
proposal.sort(key=lambda c: c["pid"])

# ── Override hook: after eyeballing, set e.g. SELECTED_PAGES = ["p0002", "p0341", ...]
SELECTED_PAGES = None
sel = SELECTED_PAGES or [c["pid"] for c in proposal]
cat = {c["pid"]: c["cat"] for c in proposal}
print(SELECTION_RULE); print()
print(f"{'page':7} {'category':14} {'words':>6} {'figs':>4}  size")
for c in proposal:
    print(f"{c['pid']:7} {c['cat']:14} {c['words']:>6} {c['figures']:>4}  {c['w']}x{c['h']}")
print(f"\nselected n={len(sel)}: {sel}")


In [ ]:
# ── 6. Render the selected pages EXACTLY as the loader would (300 DPI, RGB JPEG q80) ──
from doc_agent.ingest import loader  # noqa: E402
OUTP = Path("/kaggle/working/out/grading_kit/heldout_pages"); OUTP.mkdir(parents=True, exist_ok=True)
_idx = {c["pid"]: c["idx"] for c in census}
for pid in sel:
    target = OUTP / f"{pid}.jpg"
    if not target.exists():
        loader._atomic_render(doc[_idx[pid]], target, 300, 80)  # the repo's own render fn
print("rendered:", sorted(p.name for p in OUTP.iterdir()))

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

def show(pid, width=9):
    """Re-display any held-out page while correcting (e.g. show("p0341", width=12))."""
    img = mpimg.imread(str(OUTP / f"{pid}.jpg"))
    h, w = img.shape[:2]
    plt.figure(figsize=(width, width * h / w)); plt.imshow(img); plt.axis("off")
    plt.title(f"{pid}  ({cat.get(pid, '?')})"); plt.show()

for pid in sel:
    show(pid)


In [ ]:
# ── 7a. Drafts from DocAI (reading order, grouped into lines for easy correction) ──
def draft_text(pid):
    rows = ref_words.get(pid, [])
    if not rows:
        return ""
    hs = sorted(w["bbox_norm"][3] - w["bbox_norm"][1] for w in rows)
    med_h = hs[len(hs) // 2]
    lines, cur, last_y = [], [], None
    for w in rows:  # file order — the reference contract's source order
        y = (w["bbox_norm"][1] + w["bbox_norm"][3]) / 2
        if last_y is not None and abs(y - last_y) > med_h * 0.8 and cur:
            lines.append(cur); cur = []
        cur.append(w["text"].strip()); last_y = y
    if cur:
        lines.append(cur)
    return "\n".join(" ".join(line) for line in lines)

TQ = chr(34) * 3
print("Copy EVERYTHING between the markers into the CORRECTIONS cell below, then edit.")
print("#" * 78)
print("CORRECTIONS = {")
for pid in sel:
    body = draft_text(pid).replace("\\", "\\\\")
    assert TQ not in body, f"{pid}: draft contains a triple-quote — edit that page's draft by hand"
    print(f'    "{pid}": {TQ}\\')
    print(body + TQ + ",")
    print()
print("}")
print("#" * 78)


## ✍️ Section 7b — YOUR manual verification (this is the ground truth)

Paste the printed `CORRECTIONS = {...}` template into the next cell, then **for every page**:

1. `show("pNNNN", width=12)` in a spare cell to view the scan large (zoom with width=16 if needed).
2. Read the page and **fix every difference** between the draft and what is actually printed.
3. Watch especially for: digit-in-word swaps (`8mart-weed`→`Smart-weed` only if the page
   *actually prints* Smart-weed!), ligatures (`æ/œ`), hyphenation at line ends, small caps,
   headers/footers/page numbers DocAI may have dropped, and figure captions.
4. Blank/plate pages with no printed text keep `""` — that is a real, correct label.
5. This takes a while. It is the deliverable — the score in Section 8 is only as honest as this pass.


In [ ]:
# ── 7b. PASTE THE TEMPLATE HERE, THEN EDIT AGAINST THE SCANS ──────────────────
CORRECTIONS = {}  # ← replace with the template from 7a, corrected page by page


In [ ]:
# ── 7c. Freeze labels: validate + write grading_kit outputs ───────────────────
missing = [p for p in sel if p not in CORRECTIONS]
assert not missing, f"Section 7b incomplete — no corrected text for: {missing}"
extra = [p for p in CORRECTIONS if p not in sel]
assert not extra, f"CORRECTIONS has pages outside the frozen selection: {extra}"

OUT = Path("/kaggle/working/out/grading_kit")
labels_path = OUT / "labels.jsonl"
with open(labels_path, "w", encoding="utf-8") as f:
    for pid in sel:
        f.write(json.dumps({"page_id": pid, "text": CORRECTIONS[pid]}, ensure_ascii=False) + "\n")

rows = [json.loads(line) for line in open(labels_path, encoding="utf-8")]
assert [r["page_id"] for r in rows] == list(sel)
assert all(isinstance(r["text"], str) for r in rows)
assert all((OUT / "heldout_pages" / (r["page_id"] + ".jpg")).exists() for r in rows)
print(f"labels.jsonl written and parse-checked — {len(rows)} pages:")
for r in rows:
    print(f"  {r['page_id']}: {len(r['text'])} chars")


In [ ]:
# ── 8. Score the repo's DEFAULT pipeline (projection layout + Tesseract) ──────
import re
from collections import Counter
from doc_agent import config as config_mod  # noqa: E402
from doc_agent.contracts import Page        # noqa: E402

cfg = config_mod.load("/kaggle/working/repo/configs/config.yaml")   # defaults untouched
cfg["page_images"] = {pid: str(OUTP / f"{pid}.jpg") for pid in sel}
pages = [Page(id=pid, image_path=cfg["page_images"][pid], doc_id="pierce-1890") for pid in sel]

regions = layout_mod.detect(pages, cfg)          # default: projection
chunks = ocr_mod.transcribe(regions, cfg)        # default: tesseract
hyp = {pid: "" for pid in sel}
for ch in chunks:
    hyp[ch.page_ids[0]] = (hyp[ch.page_ids[0]] + " " + ch.text).strip()

def norm(s):
    return re.sub(r"\s+", " ", s).strip()

def lev(a, b):
    if len(a) < len(b):
        a, b = b, a
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[-1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]

def word_f1(h, r):
    H, R = Counter(h), Counter(r)
    tp = sum((H & R).values())
    if not h and not r:
        return 1.0
    if tp == 0:
        return 0.0
    p, rc = tp / sum(H.values()), tp / sum(R.values())
    return 2 * p * rc / (p + rc)

page_cer = {}
tot_ce = tot_we = tot_chars = tot_words = 0
print(f"{'page':7} {'cat':14} {'ref-chars':>9} {'CER':>7} {'WER':>7} {'wordF1':>7}")
per_page = []
for pid in sel:
    ref, h = norm(CORRECTIONS[pid]), norm(hyp[pid])
    rw, hw = ref.split(), h.split()
    ce, we = lev(h, ref), lev(hw, rw)
    cer = ce / len(ref) if ref else (0.0 if not h else float("inf"))
    wer = we / len(rw) if rw else (0.0 if not hw else float("inf"))
    f1 = word_f1(hw, rw)
    page_cer[pid] = cer
    per_page.append(dict(page_id=pid, category=cat.get(pid), ref_chars=len(ref),
                         cer=cer, wer=wer, word_f1=f1))
    tot_ce += ce; tot_we += we; tot_chars += len(ref); tot_words += len(rw)
    print(f"{pid:7} {cat.get(pid, '?'):14} {len(ref):>9} {cer:>7.3f} {wer:>7.3f} {f1:>7.3f}")

AGG = dict(micro_cer=tot_ce / max(1, tot_chars), micro_wer=tot_we / max(1, tot_words),
           n_pages=len(sel), ref_chars=tot_chars, ref_words=tot_words)
print(f"\nAGGREGATE (micro, Tesseract vs hand-verified labels): "
      f"CER={AGG['micro_cer']:.4f}  WER={AGG['micro_wer']:.4f}  over {AGG['n_pages']} pages, "
      f"{tot_chars:,} ref chars / {tot_words:,} ref words")
print("note: empty-reference pages contribute insertions to the numerator only (standard micro scoring)")

with open("/kaggle/working/out/metrics_tesseract.json", "w") as f:
    json.dump(dict(engine=TESSERACT_VERSION, layout="projection (repo default)",
                   commit=PIN, aggregate=AGG, per_page=per_page,
                   selection_rule=SELECTION_RULE), f, indent=2)
print("saved: out/metrics_tesseract.json")


In [ ]:
# ── 8b (optional). Score the offline DocAI reference mode — WITH BIAS CAVEAT ──
# CAVEAT: these labels were *seeded* from DocAI drafts, then hand-corrected. Any error
# DocAI made that the human failed to notice is invisible here, so this score is a
# LOWER BOUND on DocAI's true error. The Tesseract number above is the independent headline.
cfg_ref = {**cfg, "ocr": {**cfg["ocr"], "mode": "document_ai_reference",
                          "words_path": docai_path, "reference_missing": "empty"}}
chunks_ref = ocr_mod.transcribe(regions, cfg_ref)
hyp_ref = {pid: "" for pid in sel}
for ch in chunks_ref:
    hyp_ref[ch.page_ids[0]] = (hyp_ref[ch.page_ids[0]] + " " + ch.text).strip()
tc = te = 0
for pid in sel:
    ref, h = norm(CORRECTIONS[pid]), norm(hyp_ref[pid])
    te += lev(h, ref); tc += len(ref)
print(f"DocAI-reference mode micro CER = {te / max(1, tc):.4f}  (biased low — see caveat)")


In [ ]:
# ── 9. The worst failure (for the report's failure-case section) ──────────────
worst = max(sel, key=lambda p: (page_cer[p] if page_cer[p] != float("inf") else 9e9))
print(f"worst page by CER: {worst}  (CER={page_cer[worst]:.3f}, category={cat.get(worst)})")
show(worst, width=12)
print("── REFERENCE (yours) ──"); print(norm(CORRECTIONS[worst])[:600])
print("── TESSERACT ──");        print(norm(hyp[worst])[:600])


In [ ]:
# ── 10. Package everything for the repo ───────────────────────────────────────
import shutil
zip_path = shutil.make_archive("/kaggle/working/heldout_output", "zip", "/kaggle/working/out")
summary = dict(
    branch="a2/heldout-ocr", base_commit=PIN, n_pages=len(sel), pages=sel,
    categories={pid: cat.get(pid) for pid in sel},
    tesseract=TESSERACT_VERSION, aggregate=AGG,
    labels_convention="v1: as-printed; long-s->s; hyphens kept; whitespace-collapsed scoring; unreadable->\u2370",
)
print(json.dumps(summary, indent=2))
print()
print("DONE. Download from the right panel → Output:")
print("  heldout_output.zip   (grading_kit/heldout_pages/*.jpg + labels.jsonl + metrics json)")
print("Hand the zip + the summary JSON above back to Claude to commit into the repo.")
